# Paper Benchmarks: MST vs SDT in Incremental Datalog Evaluation

Reproducing and extending the benchmarks from the ICLP'26 submission.

**Strategies compared:**
- **Semi-naive**: Full materialization via `poll()` + `query()`
- **MST**: Magic Sets Transformation (query-directed, demand-driven)
- **SDT**: Subsumptive Demand Transformation (Tekle & Liu 2011, subsumption-aware MST)

**Evaluation modes:**
- Single-shot (fresh runtime per query)
- Incremental (persistent transformed runtime, delta-driven via `IncrementalQueryView`)

All benchmarks use the Free Join evaluation backend (Wang et al. 2024).

**Datasets:**
- RAND1K: Dense random graph (~1000 edges, 316 nodes)
- RMAT1K: Sparse RMAT graph (10,000 edges, 1000 nodes, power-law)
- LUBM1: Lehigh University Benchmark (100K+ triples)

**Programs:**
- Linear TC, Nonlinear TC
- OWL2RL entailment (128 rules)

**Query patterns:** BB (all bound), BF (one bound), FF (all free)

In [ ]:
import sys
import time
from collections.abc import Callable
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from pymycrodatalog import IncrementalQueryView, MicroRuntime, Variable

# Import data generators and rule definitions
sys.path.insert(0, str(Path(".").resolve()))
from generate import graph_to_facts, lubm_to_binary_facts, rand1k, rmat1k, split_facts
from rules import OWL2RL_ALL_RULES, TC_RULES

X = Variable("X")
Y = Variable("Y")
Z = Variable("Z")

In [ ]:
LINEAR_TC = [
    (("T", (X, Y)), ("E", (X, Y))),
    (("T", (X, Z)), ("E", (X, Y)), ("T", (Y, Z))),
]

NONLINEAR_TC = [
    (("T", (X, Y)), ("E", (X, Y))),
    (("T", (X, Z)), ("T", (X, Y)), ("T", (Y, Z))),
]

# Convert str-variable rules from rules.py to Variable-based rules
def convert_rule(rule: tuple) -> tuple:
    def convert_term(t):
        if isinstance(t, str) and t[0].isupper() and len(t) <= 2:
            return Variable(t)
        return t
    def convert_atom(atom: tuple) -> tuple:
        pred, terms = atom
        return (pred, tuple(convert_term(t) for t in terms))
    return tuple(convert_atom(a) for a in rule)

OWL2RL = [convert_rule(r) for r in OWL2RL_ALL_RULES]
print(f"OWL2RL: {len(OWL2RL)} rules")

## Helpers

In [ ]:
def bench_semi_naive(
    rules: list, facts: list, pred: str, pattern: tuple, iterations: int = 10
) -> tuple[float, int]:
    times = []
    count = 0
    for _ in range(iterations):
        rt = MicroRuntime(rules, engine="free_join")
        for f in facts:
            rt.insert(f)
        t0 = time.perf_counter_ns()
        rt.poll()
        results = rt.query(pred, pattern)
        elapsed = (time.perf_counter_ns() - t0) / 1000
        count = len(results)
        times.append(elapsed)
    times.sort()
    return times[len(times) // 2], count


def bench_strategy(
    rules: list, facts: list, pred: str, pattern: tuple, strategy: str, iterations: int = 10
) -> tuple[float, int]:
    times = []
    count = 0
    for _ in range(iterations):
        rt = MicroRuntime(rules, engine="free_join")
        for f in facts:
            rt.insert(f)
        t0 = time.perf_counter_ns()
        results = rt.query_program(pred, pattern, rules, strategy)
        elapsed = (time.perf_counter_ns() - t0) / 1000
        count = len(results)
        times.append(elapsed)
    times.sort()
    return times[len(times) // 2], count


def bench_incremental(
    rules: list, batches: list[list], pred: str, pattern: tuple, strategy: str
) -> list[dict]:
    view = IncrementalQueryView(rules, pred, pattern, strategy=strategy, engine="free_join")
    rows = []
    cumulative = 0
    for batch_idx, batch in enumerate(batches):
        for f in batch:
            view.insert(f)
        cumulative += len(batch)
        t0 = time.perf_counter_ns()
        view.poll()
        results = view.query()
        elapsed = (time.perf_counter_ns() - t0) / 1000
        rows.append({"batch": batch_idx, "facts": cumulative, "results": len(results), "time_us": elapsed})
    return rows


def bench_incremental_naive(
    rules: list, batches: list[list], pred: str, pattern: tuple
) -> list[dict]:
    rt = MicroRuntime(rules, engine="free_join")
    rows = []
    cumulative = 0
    for batch_idx, batch in enumerate(batches):
        for f in batch:
            rt.insert(f)
        cumulative += len(batch)
        t0 = time.perf_counter_ns()
        rt.poll()
        results = rt.query(pred, pattern)
        elapsed = (time.perf_counter_ns() - t0) / 1000
        rows.append({"batch": batch_idx, "facts": cumulative, "results": len(results), "time_us": elapsed})
    return rows


def run_all(label: str, rules: list, facts: list, pred: str, pattern: tuple, iters: int = 10) -> pd.DataFrame:
    rows = []
    for name, fn in [("Semi-naive", lambda: bench_semi_naive(rules, facts, pred, pattern, iters)),
                     ("MST", lambda: bench_strategy(rules, facts, pred, pattern, "Bottom-up", iters)),
                     ("SDT", lambda: bench_strategy(rules, facts, pred, pattern, "SDT", iters))]:
        t, c = fn()
        rows.append({"strategy": name, "time_us": t, "results": c})
    df = pd.DataFrame(rows)
    df.attrs["label"] = label
    return df


def make_batches(facts: list, batch_size: int) -> list[list]:
    return [facts[i:i+batch_size] for i in range(0, len(facts), batch_size)]

## 1. Load datasets

In [ ]:
rand1k_facts = graph_to_facts(rand1k())
rmat1k_facts = graph_to_facts(rmat1k())
lubm1_facts = lubm_to_binary_facts("lubm1")

print(f"RAND1K: {len(rand1k_facts)} edges")
print(f"RMAT1K: {len(rmat1k_facts)} edges")
print(f"LUBM1:  {len(lubm1_facts)} binary facts")

## 2. Single-shot: Linear TC on RAND1K and RMAT1K

Three query patterns: BB (all bound), BF (one bound), FF (all free)

In [ ]:
# Find a representative node for BF/BB queries
from collections import Counter
rand_sources = Counter(f[1][0] for f in rand1k_facts)
rand_node = rand_sources.most_common(1)[0][0]
rand_targets = [f[1][1] for f in rand1k_facts if f[1][0] == rand_node]
rand_target = rand_targets[0] if rand_targets else 0

rmat_sources = Counter(f[1][0] for f in rmat1k_facts)
rmat_node = rmat_sources.most_common(1)[0][0]
rmat_targets = [f[1][1] for f in rmat1k_facts if f[1][0] == rmat_node]
rmat_target = rmat_targets[0] if rmat_targets else 0

print(f"RAND1K query node: {rand_node} (degree {rand_sources[rand_node]}), target: {rand_target}")
print(f"RMAT1K query node: {rmat_node} (degree {rmat_sources[rmat_node]}), target: {rmat_target}")

In [ ]:
tc_results = []
for dataset_name, facts, node, target in [
    ("RAND1K", rand1k_facts, rand_node, rand_target),
    ("RMAT1K", rmat1k_facts, rmat_node, rmat_target),
]:
    for pattern_name, pattern in [
        ("BB", (node, target)),
        ("BF", (node, None)),
        ("FF", (None, None)),
    ]:
        label = f"LinTC {dataset_name} {pattern_name}"
        iters = 3 if pattern_name == "FF" else 5
        df = run_all(label, LINEAR_TC, facts, "T", pattern, iters)
        tc_results.append(df)
        print(f"\n{label} ({df.iloc[0]['results']} results):")
        print(df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey="row")
for i, df in enumerate(tc_results):
    ax = axes[i // 3][i % 3]
    df.plot.bar(x="strategy", y="time_us", ax=ax, legend=False, color="steelblue")
    ax.set_title(df.attrs["label"], fontsize=10)
    ax.set_ylabel("Time (\u00b5s)" if i % 3 == 0 else "")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
fig.suptitle("Linear TC: Semi-naive vs MST vs SDT", fontsize=14)
fig.tight_layout()
plt.show()

## 3. Nonlinear TC on RAND1K

In [ ]:
for pattern_name, pattern in [("BF", (rand_node, None)), ("FF", (None, None))]:
    label = f"NonlinTC RAND1K {pattern_name}"
    df = run_all(label, NONLINEAR_TC, rand1k_facts, "T", pattern, iters=3)
    print(f"\n{label} ({df.iloc[0]['results']} results):")
    print(df.to_string(index=False))

## 4. OWL2RL on LUBM1

128 rules, 100K+ facts. This is the knowledge base reasoning benchmark.

In [ ]:
# Semi-naive only (MST/SDT on 128 rules would be very slow for initial test)
rt = MicroRuntime(OWL2RL, engine="free_join")
for f in lubm1_facts:
    rt.insert(f)
t0 = time.perf_counter_ns()
rt.poll()
elapsed_ms = (time.perf_counter_ns() - t0) / 1e6
print(f"OWL2RL semi-naive: {elapsed_ms:.1f} ms, {len(rt)} total facts")

# Sample query: find all professors
profs = rt.query("professor", (None, None))
print(f"Professors: {len(profs)}")

students = rt.query("student", (None, None))
print(f"Students: {len(students)}")

employees = rt.query("employee", (None, None))
print(f"Employees: {len(employees)}")

## 5. Incremental evaluation: RMAT1K with streaming batches

Insert edges in batches and measure per-batch latency.
Compare: Semi-naive (incremental), MST (incremental), SDT (incremental).

In [ ]:
# Use first 2000 edges of RMAT1K for tractable incremental benchmark
rmat_subset = rmat1k_facts[:2000]
batches = make_batches(rmat_subset, 100)
print(f"{len(batches)} batches of 100 edges, {len(rmat_subset)} total")

incr = {}
incr["Semi-naive"] = bench_incremental_naive(LINEAR_TC, batches, "T", (rmat_node, None))
incr["MST (incr)"] = bench_incremental(LINEAR_TC, batches, "T", (rmat_node, None), "MST")
incr["SDT (incr)"] = bench_incremental(LINEAR_TC, batches, "T", (rmat_node, None), "SDT")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for name, rows in incr.items():
    df = pd.DataFrame(rows)
    ax1.plot(df["facts"], df["time_us"], marker=".", label=name, markersize=3)
    ax2.plot(df["facts"], df["results"], marker=".", label=name, markersize=3)
ax1.set_xlabel("Cumulative edges")
ax1.set_ylabel("Per-batch time (\u00b5s)")
ax1.set_title("Per-batch latency")
ax1.legend()
ax1.set_yscale("log")
ax2.set_xlabel("Cumulative edges")
ax2.set_ylabel("Result count")
ax2.set_title("Results over time")
ax2.legend()
fig.suptitle(f"Incremental TC BF on RMAT1K (node {rmat_node}), batch=100", fontsize=13)
fig.tight_layout()
plt.show()

## 6. Batch size effect

Same dataset, different batch sizes. Paper's key finding: smaller batches + larger cumulation = greater MST advantage.

In [ ]:
batch_sizes = [50, 100, 200, 500]
batch_size_results = {}

rmat_1k = rmat1k_facts[:1000]  # Use 1000 edges for batch size comparison

for bs in batch_sizes:
    batches = make_batches(rmat_1k, bs)
    mst_rows = bench_incremental(LINEAR_TC, batches, "T", (rmat_node, None), "MST")
    total_time = sum(r["time_us"] for r in mst_rows)
    batch_size_results[bs] = total_time
    print(f"batch_size={bs}: {len(batches)} batches, total MST time={total_time:.0f} \u00b5s")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([str(bs) for bs in batch_sizes], [batch_size_results[bs] for bs in batch_sizes], color="steelblue")
ax.set_xlabel("Batch size")
ax.set_ylabel("Total MST time (\u00b5s)")
ax.set_title("Effect of batch size on incremental MST (RMAT1K, 1000 edges)")
plt.tight_layout()
plt.show()

## 7. Summary table

In [ ]:
summary_rows = []
for df in tc_results:
    row = {"benchmark": df.attrs["label"], "results": int(df.iloc[0]["results"])}
    for _, r in df.iterrows():
        row[r["strategy"]] = f"{r['time_us']:.0f}"
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False))